# Granular Uncertainty Modeling for Reliable Left-Ventricle Segmentation and LVEF Estimation in 2D Echocardiography

**Stage 1 — Supervised baseline, clinical-endpoint validation, and granular uncertainty analysis on the CAMUS dataset**

*Author:* Iftekhar Alam Fahim, Sichuan University · *Environment:* Kaggle, NVIDIA T4 ×2 · *Code:* 

## Abstract
Automated left-ventricle (LV) segmentation in 2D echocardiography achieves high mean accuracy, yet per-case reliability remains unquantified: traditional networks emit point predictions even where acoustic shadowing and speckle noise leave the boundary undefined. This notebook (i) establishes a supervised U-Net [1] baseline with a ResNet-34 [2] encoder on the official CAMUS splits [3], (ii) validates the clinical endpoint—Left Ventricular Ejection Fraction (LVEF) by Simpson's biplane method per ASE recommendations [4]—against expert reference values, and (iii) introduces a granular-computing layer [5,6] that converts per-pixel predictive entropy into information granules of boundary confidence. Two primary research questions are evaluated: RQ1—Predictive entropy concentrates predictably at anatomical boundaries and artifact regions, particularly within expert-rated poor-quality subgroups; RQ2—Granule-level statistics reliably predict per-case segmentation and LVEF error, enabling an automated reliability gating mechanism. This granular framework provides the state-space formulation for subsequent sequential tracking agents.

*Keywords:* Echocardiography · Segmentation · Uncertainty · Granular Computing · Information Granules · LVEF

## Data availability
CAMUS [3], used under its license terms as a private pack ( `camus-v1` ); official training/validation/testing subgroups used verbatim.
**References.**  
[1] Ronneberger et al., MICCAI 2015.  
[2] He et al., CVPR 2016.  
[3] Leclerc et al., IEEE TMI 38(9):2198–2210, 2019.   
[4] Lang et al., JASE 28(1):1–39, 2015.  
[5] Pedrycz, *Granular Computing*, CRC Press, 2005.  
[6] Zadeh, Inf. Control 8(4):338–353, 1965.

## Setup and Reproducibility

In [1]:
import platform
import random
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch

# Fixed seed across all random number generators
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Strict PyTorch CUDA determinism flags
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Environment Paths
BASE = Path("/kaggle/input/datasets/brainsect/camus-v1")
NIFTI = BASE / "database_nifti"
SPLIT = BASE / "database_split"


def environment_fingerprint() -> str:
    """One-line runtime record, attached to every number this notebook publishes."""
    device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    return (
        f"python {platform.python_version()} | torch {torch.__version__} | "
        f"nibabel {nib.__version__} | numpy {np.__version__} | {device}"
    )

def frame_path(patient: Path, view: str, instant: str, mask: bool = False) -> Path:
    """Canonical frame location, tolerant of Kaggle-side gunzip (.nii vs .nii.gz)."""
    stem = f"{patient.name}_{view}_{instant}{'_gt' if mask else ''}"
    for ext in (".nii.gz", ".nii"):
        candidate = patient / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"{stem}(.nii.gz|.nii) not in {patient}")


print(environment_fingerprint())

python 3.12.13 | torch 2.10.0+cu128 | nibabel 5.4.2 | numpy 2.0.2 | Tesla T4


## Cohort Audit: completeness, geometry, spacing, label vocabulary

In [2]:
def cohort_audit_table(nifti_dir: Path) -> pd.DataFrame:
    """One row per frame; raises if any image/mask pair is missing.
    
    Headers only: nibabel defers voxel decompression, so 2,000 frames take seconds.
    """
    rows = []
    patients = sorted(p for p in nifti_dir.iterdir() if p.is_dir())

    for patient in patients:
        for view in ("2CH", "4CH"):
            for instant in ("ED", "ES"):
                img_path = frame_path(patient, view, instant)
                header = nib.load(img_path).header
                (h, w) = header.get_data_shape()[:2]
                (sx, sy) = header.get_zooms()[:2]
                
                rows.append({
                    "patient": patient.name, "view": view, "instant": instant,
                    "h": int(h), "w": int(w),
                    "sx_mm": round(float(sx), 3), "sy_mm": round(float(sy), 3)
                })

    return pd.DataFrame(rows)


def label_vocabulary(nifti_dir: Path, n: int = 20) -> set[int]:
    """Union of ground-truth labels over all four masks of a fixed-seed patient sample."""
    vocab: set[int] = set()
    patients = sorted(p for p in nifti_dir.iterdir() if p.is_dir())

    for patient in random.Random(SEED).sample(patients, n):
        for view in ("2CH", "4CH"):
            for instant in ("ED", "ES"):
                gt_path = frame_path(patient, view, instant, mask=True)
                mask = np.squeeze(np.asarray(nib.load(gt_path).dataobj))
                vocab |= set(np.unique(mask).astype(int).tolist())

    return vocab


# Audit execution: completeness, geometry, and labels
audit_table = cohort_audit_table(NIFTI)
vocab = label_vocabulary(NIFTI, n=20)

print(f"{audit_table.patient.nunique()} patients, {len(audit_table)} frames | "
      f"Spacings: {audit_table[['sx_mm', 'sy_mm']].drop_duplicates().values.tolist()} mm | "
      f"Labels: {sorted(vocab)}")

audit_table.groupby("patient")[["h", "w"]].first().describe().loc[["min", "mean", "max"]].round(0)

500 patients, 2000 frames | Spacings: [[0.308, 0.308]] mm | Labels: [0, 1, 2, 3]


,h,w
min,323.0,292.0
mean,601.0,492.0
max,1181.0,973.0


## Official Subgroups: sizes, overlap structure, coverage, contract.

In [3]:
def read_subgroup(filename: str) -> list[str]:
    """Patient identifiers of one official subgroup file, in file order."""
    filepath = SPLIT / filename
    if not filepath.exists():
        raise FileNotFoundError(f"Subgroup file not found: {filepath}")
    return [
        ln.strip() for ln in filepath.read_text().splitlines() if ln.strip()
    ]


# Read Official subgroups, verbatim from the release
train_ids = read_subgroup("subgroup_training.txt")
val_ids = read_subgroup("subgroup_validation.txt")
test_ids = read_subgroup("subgroup_testing.txt")
train_set, val_set, test_set = set(train_ids), set(val_ids), set(test_ids)

# Invariant: No duplicates within individual subgroup files
assert (
    len(train_ids) == len(train_set)
    and len(val_ids) == len(val_set)
    and len(test_ids) == len(test_set)
), "Duplicate patient identifiers detected inside a subgroup file!"

# Overlap structure audit
print(
    f"overlaps | train-val: {len(train_set & val_set)} | "
    f"train-test: {len(train_set & test_set)} | val-test: {len(val_set & test_set)}"
)

# Strict evaluation contract: Test set must never overlap with train or val
assert (
    not (train_set & test_set) and not (val_set & test_set)
), "Test subgroup contaminated! Test set overlaps with train or val."

# Bi-directional coverage audit (Disk vs. Subgroup Manifests)
disk_patients = {
    p.name
    for p in NIFTI.iterdir()
    if p.is_dir() and p.name.startswith("patient")
}
listed_patients = train_set | val_set | test_set

print(
    f"On disk but unlisted : {sorted(disk_patients - listed_patients) or 'None'}"
)
print(
    f"Listed but absent disk: {sorted(listed_patients - disk_patients) or 'None'}"
)

# Summary table relative to listed cohort
subgroup_df = pd.DataFrame(
    {
        "subgroup": ["train", "val", "test"],
        "n_patients": [len(train_ids), len(val_ids), len(test_ids)],
        "fraction_of_listed": [
            round(len(s) / len(listed_patients), 3)
            for s in (train_set, val_set, test_set)
        ],
    }
)

print("\n" + subgroup_df.to_string(index=False))
print(f"Sample identifier format: {train_ids[:3]}")

overlaps | train-val: 0 | train-test: 0 | val-test: 0
On disk but unlisted : None
Listed but absent disk: None

subgroup  n_patients  fraction_of_listed
   train         400                 0.8
     val          50                 0.1
    test          50                 0.1
Sample identifier format: ['patient0001', 'patient0002', 'patient0003']
